# 02 - Extração → Bronze
Baixa os dados das APIs para o Volume `raw` (JSON bruto) e depois cria as tabelas Bronze.

| Fonte | Caminho no Volume |
|---|---|
| Times | `raw/football/teams/season=2026/teams.json` |
| Partidas | `raw/football/matches/season=2026/matchday=NN/matches.json` |
| Clima | `raw/weather/season=2026/cidade_id=<cidade>/weather.json` |

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import logging
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
logging.getLogger("src").setLevel(logging.INFO)

from src.config import load_settings
from src.extract import football
from src.raw_storage import read_json

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()
cliente_futebol = football.build_client(settings)

In [0]:
times = read_json(football.extract_teams(cliente_futebol, settings))
print("Temporada:", times["season"]["startDate"], "→", times["season"]["endDate"])
for t in sorted(times["teams"], key=lambda t: t["name"]):
    print(t["id"], "|", t["name"], "|", t.get("venue"), "|", t.get("address"))

In [0]:
resumo = football.extract_matches(cliente_futebol, settings)
print(resumo)

In [0]:
display(dbutils.fs.ls(f"{settings.raw_path}/football/matches/season={settings.season}"))

In [0]:
from datetime import date
from pathlib import Path

from src.extract import weather

times = read_json(football.football_dir(settings) / "teams" / f"season={settings.season}" / "teams.json")
estadios = weather.load_stadiums(Path("..") / "reference" / "estadios.csv")
inicio, fim = weather.weather_period(
    date.fromisoformat(times["season"]["startDate"]),
    date.fromisoformat(times["season"]["endDate"]),
)
print(f"{len(estadios)} times | período do clima: {inicio} → {fim}")

resumo_clima = weather.extract_weather(weather.build_client(settings), settings, estadios, inicio, fim)
print(resumo_clima)